好的，这是根据您提供的 PPT 内容生成的详细 Markdown 笔记。

-----

# 讲座 16: 从可验证奖励中进行强化学习 (RLVR) 笔记

## 目录

1.  **从 RLHF 到 DPO: 寻求更简洁的方案**
      * RLHF 的基本流程
      * DPO：无需奖励模型的 RLHF
      * DPO 的推导过程
      * DPO 的梯度更新分析
2.  **实践中的强化学习算法**
      * PPO (Proximal Policy Optimization)
          * 理论与目标
          * 实践中的复杂性
          * 语言模型中的 PPO 实现细节
      * GRPO (Group Relative Policy Optimization)
          * 核心思想：简化 PPO
          * GRPO 的偏见与改进 (Dr. GRPO)
3.  **RLVR 案例研究**
      * Deepseek R1
          * 算法、设置与关键发现
          * SFT 初始化与 RL 阶段
          * 失败的尝试：PRM 和 MCTS
      * Kimi K1.5
          * 数据筛选与 SFT
          * 独特的 RL 算法
          * 长度控制与系统架构
      * Qwen 3
          * SFT 与低数据量 RL
          * 思维模式融合
          * 分阶段训练的效果

-----

## 1\. 从 RLHF 到 DPO: 寻求更简洁的方案

### RLHF (Reinforcement Learning from Human Feedback) 的基本流程

[cite\_start]RLHF 是一种利用人类偏好数据来微调语言模型的方法 [cite: 12]。其标准流程如下图所示：

1.  [cite\_start]**收集偏好数据**: 针对同一个提示 (prompt) $x$，采样多个模型的输出，由人类标注者选出最优的回答 $y\_w$ (winner) 和较差的回答 $y\_l$ (loser) [cite: 13, 23, 17]。
2.  [cite\_start]**训练奖励模型**: 使用这些偏好数据 $(x, y\_w, y\_l)$ 训练一个奖励模型 $r\_\\phi(x,y)$ [cite: 19]，目标是让 $r\_\\phi(x,y\_w) \> r\_\\phi(x,y\_l)$。
3.  [cite\_start]**强化学习微调**: 使用奖励模型 $r\_\\phi$ 作为奖励函数，通过 PPO 等强化学习算法对语言模型策略 $\\pi\_\\theta$ 进行优化，最大化奖励信号，同时用 KL 散度惩罚使其不过于偏离原始的参考模型 $\\pi\_{ref}$ [cite: 20, 25, 26]。

\<img src="[https://storage.googleapis.com/static.aidevs.com/gemini/production/2025-09-11/10:35:04.133748/18d8e58f.png](https://www.google.com/search?q=https://storage.googleapis.com/static.aidevs.com/gemini/production/2025-09-11/10:35:04.133748/18d8e58f.png)" alt="RLHF vs DPO Process" width="600"/\>
[cite\_start]*左图为 RLHF 流程，右图为 DPO 流程 [cite: 13, 14, 22, 27, 28, 29]。*

### DPO (Direct Preference Optimization)：无需奖励模型的 RLHF

[cite\_start]DPO 旨在简化 PPO 流程，它实现了 **"RLHF without tears"** [cite: 5]，其核心优势在于：

  * [cite\_start]**无需显式奖励模型**: DPO 直接在偏好数据上优化策略，绕过了训练独立奖励模型的步骤 [cite: 7]。
  * [cite\_start]**无需在线策略学习**: 摆脱了 PPO 中的 rollouts 和复杂的内外循环，变为一个简单的离线损失函数 [cite: 8]。

DPO 的核心思想可以概括为：

  * [cite\_start]对偏好数据中的 "good stuff" (winning response $y\_w$) 进行**对数损失的梯度上升** [cite: 10]。
  * [cite\_start]对 "bad stuff" (losing response $y\_l$) 进行**加权的梯度下降** [cite: 11]。

### DPO 的推导过程

[cite\_start]DPO 的目标函数是从 RLHF 的优化目标推导出来的 [cite: 30]。

1.  **RLHF 的优化目标**:
    [cite\_start]RLHF 的目标是最大化奖励，同时限制策略偏离参考模型 [cite: 31]。

    $$
    $$$$\\max\_{\\pi\_{\\theta}}\\mathbb{E}*{x\\sim\\mathcal{D},y\\sim\\pi*{\\theta}(y|x)}[r\_{\\phi}(x,y)]-\\beta\\mathbb{D}*{KL}[\\pi*{\\theta}(y|x)||\\pi\_{ref}(y|x)]

    $$
    $$$$
    $$
2.  **最优策略的解析解**:
    [cite\_start]假设策略空间是无参数的 (nonparametric assumption)，那么上述目标的最优策略 $\\pi\_r$ 具有以下形式 [cite: 33, 34]：

    $$
    $$$$\\pi\_{r}(y|x)=\\frac{1}{Z(x)}\\pi\_{ref}(y|x)\\exp\\left(\\frac{1}{\\beta}r(x,y)\\right)

    $$
    $$$$其中 $Z(x)$ 是归一化因子。

3.  **反解隐式奖励**:
    [cite\_start]从最优策略的表达式中，可以反解出奖励函数 $r(x,y)$ [cite: 35]：

    $$
    $$$$r(x,y)=\\beta \\log\\frac{\\pi\_{r}(y|x)}{\\pi\_{ref}(y|x)}+\\beta \\log Z(x)

    $$
    $$$$[cite\_start]这个关系是 DPO 的核心，它将奖励 $r$ 和策略 $\\pi$ 直接关联起来 [cite: 44, 45]。

4.  **构建 DPO 损失函数**:
    [cite\_start]将这个隐式奖励函数代入标准的成对比较损失 (Bradley-Terry 模型) 中 [cite: 39][cite\_start]，即最大化 $y\_w$ 的奖励高于 $y\_l$ 的概率，得到最终的 DPO 目标函数 [cite: 41, 42]：

    $$
    $$$$\\mathcal{L}*{DPO}(\\pi*{\\theta};\\pi\_{ref})=-\\mathbb{E}*{(x,y*{w},y\_{l})\\sim\\mathcal{D}}\\left[\\log\\sigma\\left(\\beta\\log\\frac{\\pi\_{\\theta}(y\_{w}|x)}{\\pi\_{ref}(y\_{w}|x)}-\\beta\\log\\frac{\\pi\_{\\theta}(y\_{l}|x)}{\\pi\_{ref}(y\_{l}|x)}\\right)\\right]

    $$
    $$$$其中 $\\sigma$ 是 sigmoid 函数。

### DPO 的梯度更新分析

[cite\_start]DPO 损失函数对模型参数 $\\theta$ 的梯度可以被分解为 [cite: 50, 51]：
$$\nabla_{\theta}\mathcal{L}_{DPO}(\pi_{\theta};\pi_{ref}) = -\beta \mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}} \left[ \sigma(\dots) \underbrace{\left( \nabla_{\theta}\log\pi_{\theta}(y_w|x) - \nabla_{\theta}\log\pi_{\theta}(y_l|x) \right)}_{\text{增加 } y_w \text{ 的概率，降低 } y_l \text{ 的概率}} \right]$$
[cite\_start]这个形式直观地体现了 DPO 的作用 [cite: 49]：

  * [cite\_start]**提高 $y\_w$ 的似然** [cite: 54]。
  * [cite\_start]**降低 $y\_l$ 的似然** [cite: 55]。
  * [cite\_start]更新的权重由 $\\sigma(\\dots)$ 项调节，该项可以理解为**隐式奖励模型的预测误差** [cite: 56][cite\_start]。当模型对 $(y\_w, y\_l)$ 的偏好判断错误时，权重会更高 [cite: 52]。

[cite\_start]实验结果表明，DPO 的性能与精心调优的 PPO 相当，但实现更为简单 [cite: 58, 59][cite\_start]。如今，大多数顶级的开源 RLHF 模型都采用了 DPO [cite: 62]。

## 2\. 实践中的强化学习算法

### PPO (Proximal Policy Optimization)

[cite\_start]PPO 是 RLHF 和更广泛的 RL 任务中常用的策略梯度方法 [cite: 130]。

#### 理论与目标

[cite\_start]PPO 通过在一个"裁剪" (clipping) 的目标函数上进行优化，来避免策略更新步子过大导致性能崩溃。其核心目标函数如下 [cite: 128]：
$$L(s, a, \theta_k, \theta) = \min\left( \frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)} A^{\pi_{\theta_k}}(s,a), \quad \text{clip}\left(\frac{\pi_\theta(a|s)}{\pi_{\theta_k}(a|s)}, 1-\epsilon, 1+\epsilon\right) A^{\pi_{\theta_k}}(s,a) \right)$$

  * $\\frac{\\pi\_\\theta(a|s)}{\\pi\_{\\theta\_k}(a|s)}$ 是重要性采样权重。
  * $A^{\\pi\_{\\theta\_k}}(s,a)$ 是**优势函数 (Advantage Function)**，表示在状态 $s$ 下采取动作 $a$ 相对于平均水平的好坏。
  * [cite\_start]`clip` 函数将概率比率限制在 $[1-\\epsilon, 1+\\epsilon]$ 范围内，通常 $\\epsilon=0.2$ [cite: 141]。

#### 实践中的复杂性

[cite\_start]尽管 PPO 理论上很优雅，但在实践中实现起来非常复杂且有很多细节 [cite: 133, 134]，包括：

  * [cite\_start]**价值模型 (Value Model)**: 需要一个额外的模型 $V\_\\phi(s)$ 来估计状态的价值，并用于计算优势函数，这会消耗大量内存并需要额外调优 [cite: 153]。
  * [cite\_start]**在线 Rollouts**: 需要在当前策略下进行采样 (rollouts) 来收集经验，这使得算法是**在线**的，效率较低 [cite: 213]。
  * [cite\_start]**奖励塑造 (Reward Shaping)**: 在语言模型中，通常会将最终的序列奖励与每一步的 KL 惩罚结合起来，以维持稳定性 [cite: 143]。
  * [cite\_start]**广义优势估计 (GAE)**: 一种更稳定的优势函数计算方法，是 PPO 的一个关键组件 [cite: 146, 147]。

\<img src="[https://storage.googleapis.com/static.aidevs.com/gemini/production/2025-09-11/10:35:04.133748/0ed12678.png](https://www.google.com/search?q=https://storage.googleapis.com/static.aidevs.com/gemini/production/2025-09-11/10:35:04.133748/0ed12678.png)" alt="PPO for LLMs" width="600"/\>
[cite\_start]*语言模型中 PPO 的训练流程图 [cite: 135]。*

### GRPO (Group Relative Policy Optimization)

[cite\_start]GRPO 是一个新兴的 RL 算法，旨在进一步简化 PPO [cite: 155]。

#### 核心思想：简化 PPO

[cite\_start]GRPO 的主要创新在于**移除了价值模型和复杂的优势函数计算** [cite: 155]。它保留了 PPO 的裁剪目标函数结构，但用一种非常简单的方式来计算优势 $A\_i$：

  * 对于一个 prompt，从当前策略中采样一组 $G$ 个输出 ${o\_1, o\_2, \\dots, o\_G}$。
  * 计算每个输出的奖励 ${r\_1, r\_2, \\dots, r\_G}$。
  * [cite\_start]优势 $A\_i$ 被定义为该组内奖励的**z-score标准化值** [cite: 155]。

[cite\_start]**GRPO 的优势函数**[cite: 155]:
$$A_i = \frac{r_i - \text{mean}(\{r_1, r_2, \dots, r_G\})}{\text{std}(\{r_1, r_2, \dots, r_G\})}$$

[cite\_start]**GRPO 的目标函数**[cite: 155]:
$$\mathcal{J}_{GRPO}(\theta) = \mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{old}}(O|q)} \left[ \frac{1}{G}\sum_{i=1}^G \min\left( \frac{\pi_\theta(o_i|q)}{\pi_{\theta_{old}}(o_i|q)}A_i, \text{clip}(\dots)A_i \right) - \beta D_{KL}(\pi_\theta||\pi_{ref}) \right]$$

#### GRPO 的偏见与改进 (Dr. GRPO)

GRPO 的优势计算虽然简单，但引入了理论上的问题：

  * [cite\_start]**有偏基线 (Biased Baseline)**: 根据策略梯度理论，从奖励中减去的基线 (baseline) 不能依赖于当前采取的动作 [cite: 164][cite\_start]。GRPO 中的均值和标准差都依赖于当前批次的所有样本，因此它不是一个有效的无偏基线 [cite: 165]。
  * [cite\_start]**长度偏见**: 分母中的标准差项可能会导致模型过度关注那些奖励方差大的问题 (无论简单还是困难)，并且实验表明它会不必要地增长不正确答案的长度 [cite: 168, 180]。

**Dr. [cite\_start]GRPO (GRPO Done Right)** 提出了修正方案，其优势函数仅减去均值，移除了有偏的标准差项，从而变成了一个无偏的估计器 [cite: 166]。
$$\hat{A}_{i,j} = R(q_i, o_{i,j}) - \text{mean}(\{R(q_i, o_{i,k})\}_{k \ne j})$$
*(注：上式接近留一法均值，是无偏的)*

## 3\. RLVR 案例研究

### Deepseek R1

[cite\_start]Deepseek R1 是一个通过 RLVR 在推理任务上取得巨大成功的模型，其开源的配方引发了广泛关注 [cite: 172, 173]。

  * [cite\_start]**核心算法**: R1 主要使用 **GRPO** 算法进行 RL 训练 [cite: 174]。
  * [cite\_start]**训练设置 (R1-zero)**[cite: 176]:
      * **奖励**: 包含两部分：1) 准确性奖励 (答案是否正确)；2) 格式奖励 (是否使用 `<think>` 标签)。
      * **基础模型**: Deepseek-V3。
  * **关键发现**:
      * [cite\_start]**"Aha" 时刻**: 训练过程中观察到模型会自行修正错误，并输出类似 "Wait, wait, wait. That's an aha moment" 的文本 [cite: 178][cite\_start]。但后续分析表明，这种能力可能在基础模型中就已存在，且 GRPO 的长度偏见可能放大了这种现象 [cite: 180, 181]。
      * [cite\_start]**CoT 长度增长**: 训练过程中，思维链 (CoT) 的平均长度显著增加 [cite: 177]。
  * [cite\_start]**完整 R1 流程**[cite: 182]:
    1.  [cite\_start]**推理 SFT**: 使用长 CoT 数据进行监督微调 [cite: 184, 187]。
    2.  [cite\_start]**RL (GRPO)**: 在 SFT 模型上进行 GRPO 训练，并加入了一个**语言一致性奖励**来避免 CoT 中出现多语言混合的问题 [cite: 185, 191]。
    3.  [cite\_start]**SFT/RLHF**: 最后阶段，在通用任务上进行传统的 SFT 和 RLHF (同样使用 GRPO) [cite: 192, 194]。
  * [cite\_start]**失败的尝试**: 报告中提到，PRM (Process Reward Models) 和 MCTS (Monte Carlo Tree Search) 等更复杂的方法效果有限，且计算开销大 [cite: 197, 198, 199]。

### Kimi K1.5

[cite\_start]Kimi K1.5 与 R1 同期发布，也通过 RLVR 实现了顶尖的性能 [cite: 200, 201]。

  * **数据筛选**:
      * [cite\_start]**难度过滤**: 只选择那些模型在 best-of-8 采样中失败的难题进行训练 [cite: 202]。
      * [cite\_start]**任务均衡**: 确保数据覆盖不同领域和学科 [cite: 202]。
  * **独特的 RL 算法**:
    [cite\_start]Kimi 团队从 DPO 的推导中获得灵感，设计了一种新的带基线的策略梯度算法 [cite: 203][cite\_start]。它将奖励函数用策略和参考模型表示，并使用平方损失作为代理损失 [cite: 204, 205][cite\_start]。最终的梯度形式如下 [cite: 206]：
    $$
    $$$$\\frac{1}{k}\\sum\_{j=1}^k \\left( \\nabla\_\\theta \\log \\pi\_\\theta(y\_j, z\_j|x) (r(x, y\_j, y^\*) - \\bar{r}) - \\frac{\\tau}{2}\\nabla\_\\theta \\left( \\log \\frac{\\pi\_\\theta(y\_j, z\_j|x)}{\\pi\_{\\theta\_k}(y\_j, z\_j|x)} \\right)^2 \\right)
    $$
    $$$$
    $$
  * **长度控制**:
    [cite\_start]为了压缩 CoT 的长度，Kimi 引入了一个显式的**长度奖励** [cite: 208, 209][cite\_start]。对于同一批次中的样本，较长的序列会获得负奖励，从而激励模型在保证正确的前提下生成更简洁的回答 [cite: 210]。
  * **系统架构**:
    [cite\_start]为了解决 RL 训练中推理 (rollout) 和训练 (gradient update) 交替导致的效率低下问题，Kimi 设计了一个混合部署框架，使用 vLLM 进行高效推理，使用 Megatron 进行训练，并通过共享内存快速切换 [cite: 212, 214]。

### Qwen 3

[cite\_start]Qwen 3 是最新的开源推理模型之一，其特点在于**数据效率极高**的 RLVR [cite: 171, 218]。

  * **SFT 与低数据量 RL**:
    [cite\_start]Qwen 3 的 RL 阶段**仅使用了 3995 个样本** [cite: 220][cite\_start]。这得益于其精细的数据筛选策略，包括难度过滤、移除简单样本以及对 CoT 质量的人工筛选 [cite: 220]。
  * [cite\_start]**思维模式融合 (Thinking Mode Fusion)**[cite: 221]:
    为了让模型能够按需生成 CoT，Qwen 3 在训练数据中混合了两种模式：
    1.  **思考模式**: `...{query}</think><im_end> ... <think>{thinking_content}</think>...`
    2.  **非思考模式**: `...{query}/no think<im_end> ... <think></think>...`
        [cite\_start]通过这种方式，模型学会了根据指令决定是否进行显式的链式思考，并能通过一个特殊的停止标记来控制思考过程的长度 [cite: 222]。
  * **分阶段训练的效果**:
    [cite\_start]报告显示，不同的训练阶段对模型能力的影响不同。例如，在经过通用的 RLHF 训练后，模型在数学和 STEM 基准上的性能略有下降，但在通用任务和指令遵循方面的能力得到了提升 [cite: 224]。

-----